# S2b: B0 (base Tiny Aya En-Thinker, `/no_think`) on one dev episode

**Before running:**
1. Settings: Accelerator = GPU T4 x1 (or P100), Internet = On.
2. Add-ons > Secrets: `HF_TOKEN` and `API_DEEPSEEK` (attach both to this notebook).
3. Add a **private** dataset containing `/tmp/agentic_distill_src.zip` (made on the Mac by `scripts/make_kaggle_bundle.sh`), named e.g. `agentic-distill-src`, and attach it.

Stops (does not work around) if Python < 3.12, no GPU, or an install fails.

In [ ]:
import sys, subprocess, os, glob
print(sys.version)
assert sys.version_info >= (3, 12), "tau2 needs Python >= 3.12; stop and report (do not hack around)"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
# Kaggle may keep the upload as a .zip or auto-extract it; accept either.
zips = glob.glob("/kaggle/input/**/agentic_distill_src.zip", recursive=True)
dirs = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
print("zip:", zips, "extracted:", dirs)
assert zips or dirs, "attach the private bundle dataset"


In [ ]:
import urllib.request
for u in ("https://huggingface.co", "https://api.deepseek.com", "https://github.com"):
    try:
        urllib.request.urlopen(urllib.request.Request(u, method="HEAD"), timeout=10); print("internet ok:", u)
    except Exception as e:
        print("internet check:", u, type(e).__name__, str(e)[:80])      # HTTP 4xx still proves connectivity
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
missing = []
for label, var in (("HF_TOKEN", "HF_TOKEN"), ("API_DEEPSEEK", "API_DEEPSEEK")):
    try:
        os.environ[var] = s.get_secret(label)
    except Exception as e:
        missing.append(label); print("secret NOT available:", label, type(e).__name__)
print("secrets loaded:", [k for k in ("HF_TOKEN", "API_DEEPSEEK") if os.environ.get(k)])   # values never printed
assert not missing, f"attach these secrets to this notebook version: {missing}"


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
Z=$(find /kaggle/input -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); \
else D=$(dirname "$(find /kaggle/input -name pyproject.toml | head -1)"); cp -r "$D"/. /kaggle/working/repo/; fi
cd /kaggle/working/repo && ls
mkdir -p third_party && git clone -q https://github.com/sierra-research/tau2-bench third_party/tau2-bench
git -C third_party/tau2-bench checkout -q c06b07dd7c4d4ec61ba6ff9bcdf0b15dd7ef36d4
pip install -q -e third_party/tau2-bench audioop-lts pyyaml transformers accelerate
python -c "import tau2, transformers, torch; print('tau2 ok; torch', torch.__version__, torch.cuda.is_available())"


In [ ]:
%%bash
cd /kaggle/working/repo
PYTHONPATH=src python -m pytest -q tests/test_formatter.py -x 2>&1 | tail -3   # skips if tokenizer dir absent; fine


In [ ]:
%%bash
cd /kaggle/working/repo
# Dev tasks 75, 91, 34 (D7 stop-string protocol). Add --allow-peak only if you accept 2x DeepSeek user-sim prices.
PYTHONPATH=src python -m agentic_distill.student_run --tasks 75 91 34 --trials 1 --slice s3_pilot --label b0 \
  --out-root /kaggle/working/out 2>&1 | grep -v DEBUG | tail -25


In [ ]:
import json, glob
for p in glob.glob("/kaggle/working/out/data/raw/*/episodes.jsonl"):
    for l in open(p): print(json.dumps(json.loads(l), indent=1))
# Download /kaggle/working/out/data/raw/<run>/ (traces + episodes.jsonl) back to the Mac for review.
